# Data Onboarding and Contracts

## What arrived at Riverside?

Six systems have delivered fourteen frozen sample records: policy and rights PDFs, manuscript text, ERP rows, workflow pages, and identity records. Every sample has an ID and a payload. That surface completeness hides the real onboarding questions.

One policy is stale. One PDF loses meaning if its columns are flattened. Another repeats a page. An autosave was deleted. An ERP row has no rights territory. An API changed a required field without changing its contract. A disabled contractor still survives in an old group snapshot. Riverside cannot answer *what may become searchable?* by looking at text alone; it also needs ownership, purpose, version, access, lifecycle, and a deletion path.

The chapter follows those facts in the order an onboarding pipeline must face them:

```mermaid
flowchart LR
    A["PDF, text, ERP, and API records"] --> B["Check owner and purpose"]
    B --> C["Map without guessing"]
    C --> D["Select current versions"]
    D --> E["Apply current access"]
    E --> F["Trace updates and deletion"]
    F --> G["Per-source readiness decision"]
```

## What the local evidence can say

Loading `RIV-FDE-1.0.0` should confirm six sources and fourteen records. Results calculated from those committed fixtures are labeled `[Measured - local fixture]`. Owner estimates and business constraints retain their original claim class; a calculation over a customer claim does not turn it into observed customer truth.

This notebook produces teaching artifacts only. It writes no production artifact and cannot validate customer completeness, Databricks behavior, current identity, legal interpretation, or operating readiness. Those boundaries stay visible as the records move toward a bounded readiness verdict.

The working rule is simple: text is not a searchable document until its authority, current state, and deletion path are reviewable.

## 0 - The Tempting Shortcut

A generic loader offers an attractive first answer: if `document_id` and `payload` exist, flatten the payload and upsert it. On this fixture that rule accepts all fourteen records. It also carries every seeded failure toward search: stale policy, scrambled meaning, unsupported rights, schema drift, deleted text, and stale authority.

```mermaid
flowchart LR
    A["6 sources and 14 records"] --> B["Naive flatten and upsert"]
    B --> C["Stale, malformed, broad, or undeletable content"]
    C --> D["Source contracts and held-for-review paths"]
    D --> E["Lifecycle and current-access checks"]
    E --> F["Per-source readiness verdict"]
```

### Predict before the run

Will the presence rule accept all fourteen records? Commit to an answer before executing the loader. Then compare that count with the smaller set left after the fixture's explicit non-index decisions.

The comparison is not a quality score. It is the first sign that transport success and search readiness answer different questions. From here, each source keeps its own lifecycle, parser, contract, deletion, and access evidence; one aggregate pass rate is not allowed to hide a material failure.

When recording an observed result, retain the environment, commit, fixture version, method, result, and limitations. The run may prove that local rules detect seeded failures. It cannot prove customer data or production readiness.

In [ ]:
# -- Load and validate the frozen case ---------------------------------------
from collections import Counter, defaultdict
from copy import deepcopy
from pathlib import Path
import json

from jsonschema import Draft202012Validator

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "AUTHORING_GUIDE.md").is_file() and (candidate / "learning" / "role-based-tracks" / "fde" / "shared").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the ai-portfolio repository.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
SHARED_DIR = REPO_ROOT / "learning" / "role-based-tracks" / "fde" / "shared"
FIXTURE_DIR = SHARED_DIR / "fixtures"
SCHEMA_DIR = SHARED_DIR / "schemas"

def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

engagement = load_json(FIXTURE_DIR / "riverside-engagement-v1.json")
samples = load_json(FIXTURE_DIR / "riverside-source-samples-v1.json")
facts = load_json(FIXTURE_DIR / "expected-facts-v1.json")
for name, value, schema_name in (
    ("engagement", engagement, "riverside-engagement.schema.json"),
    ("samples", samples, "riverside-source-samples.schema.json"),
    ("facts", facts, "expected-facts.schema.json"),
):
    errors = sorted(
        Draft202012Validator(load_json(SCHEMA_DIR / schema_name)).iter_errors(value),
        key=lambda error: list(error.path),
    )
    assert not errors, f"{name}: {errors[0].message if errors else 'schema error'}"

records = samples["records"]
print(f"Case: {engagement['fixture_id']} | version: {engagement['fixture_version']}")
print(f"Sources: {len(engagement['source_inventory'])} | sample records: {len(records)}")
print(f"Expected facts: {len(facts['facts'])}")

## Follow One Record by Hand Before Scaling Out

Before generalizing across fourteen records, walk `REC-RIV-PDF-001` through the decision path one fact at a time:

1. **Arrival:** the fixture supplies a stable record ID, source ID, document ID, and PDF payload. Presence proves only that the transport delivered something.
2. **Authority:** the source ID must resolve to the inventory entry that carries owner, approved purpose, access, refresh, retention, and deletion expectations. No display-name match or inferred owner is acceptable.
3. **Lifecycle:** this policy record is marked `superseded`. That fact is decisive before text quality or similarity is considered.
4. **Disposition:** `local_disposition` assigns `EXCLUDE_VERSION` with reason `not_current`.
5. **Search effect:** the 2024 policy stays in immutable history and lineage, but it is absent from the current searchable view. It must not produce current chunks or vectors.
6. **Evidence claim:** the path above is measured only against the frozen fixture. A Riverside owner and the target jobs must still validate source authority and downstream enforcement.

That is the full journey from *what arrived?* to *what may become searchable?* for one record: delivered does not mean current, and preserved history does not mean exposed content.

### Now widen the prediction

Predict whether the same presence rule accepts all fourteen records. The next cell automates only that deliberately naive rule and contrasts it with explicit fixture decisions. Treat its result as a transport check, not a readiness verdict.

In [ ]:
# -- Expose the naive readiness failure --------------------------------------
naive_ready = [record for record in records if record.get('document_id') and record.get('payload')]
non_index_decisions = {
    'exclude_from_current_policy_index', 'quarantine_until_duplicate_and_ocr_review',
    'exclude_and_emit_tombstone', 'quarantine_for_owner_resolution',
    'quarantine_schema_drift', 'preserve_for_incident_replay',
    'allow_with_request_purpose', 'deny_and_reconcile',
}
controlled_candidates = [record for record in records if record['expected_onboarding']['decision'] not in non_index_decisions]
print(f'[Measured - local fixture] Naive ready count: {len(naive_ready)} of {len(records)}')
print(f'[Measured - local fixture] Candidates before deeper gates: {len(controlled_candidates)}')
print('Prediction resolution: A is the naive result; B is the required control model.')

## 1 - Name the Authority Behind the Payload

The manual policy walk exposed the first dependency: a record cannot carry its own authority. Riverside needs a source ledger that names who may approve purpose, access, retention, refresh, and deletion before an adapter interprets the payload.

```mermaid
flowchart LR
    A["Frozen source inventory"] --> B["Join records by stable source ID"]
    B --> C{"Owner, purpose, access, refresh, and deletion known?"}
    C -->|"No"| D["Block and assign an owner"]
    C -->|"Yes"| E["Sample the approved scope"]
    E --> F["DATA-01"]
```

The join uses stable source IDs, not display names. That matters when an alias changes or rows arrive in a different order; a friendly label must never attach a record to the wrong owner. Estimated counts and freshness targets also remain customer claims until the named owners validate them.

### Stress the sampling rule

Change `required_samples_per_source` from one to three. Which sources now have shortfalls? The count is measured against the fixture, but the threshold of three is an invented policy choice unless an owner approves it. Keep those two claims separate in DATA-01.

The inventory gate closes only when each approved scope has an owner and a replayable source identity. More rows cannot compensate for missing authority.

In [ ]:
# -- Build and check DATA-01 -------------------------------------------------
inventory = {source['source_id']: source for source in engagement['source_inventory']}
records_by_source = defaultdict(list)
for record in records:
    records_by_source[record['source_id']].append(record)
orphan_ids = sorted(record['record_id'] for record in records if record['source_id'] not in inventory)
unsampled_ids = sorted(set(inventory) - set(records_by_source))
source_types = sorted({source['source_type'] for source in inventory.values()})
required_samples_per_source = 1  # CHANGE THIS: try 3, then justify it.
sample_shortfalls = {source_id: required_samples_per_source - len(records_by_source[source_id]) for source_id in inventory if len(records_by_source[source_id]) < required_samples_per_source}
assert samples['fixture_version'] == engagement['fixture_version'] == 'RIV-FDE-1.0.0'
assert not orphan_ids and not unsampled_ids
assert set(source_types) == {'API', 'ERP', 'PDF', 'text'}
assert all(source['owner_person_id'] for source in inventory.values())
print(f'[Measured - local fixture] Sources joined: {len(inventory)}; types: {source_types}')
print(f'[Measured - local fixture] Orphans: {orphan_ids}; unsampled: {unsampled_ids}')
print(f'Policy result at sample threshold {required_samples_per_source}: {sample_shortfalls}')
print('[Customer claim retained] Counts, freshness, ownership, and deletion behavior remain unvalidated.')

## 2 - Give Each Source a Safe Exit

Once authority is attached, payloads stop looking interchangeable. A two-column PDF needs layout-aware parsing; repeated or low-confidence pages need review; a deleted text record needs a tombstone; and missing business fields need an owner, not a guessed default.

```mermaid
flowchart LR
    A["PDF, text, ERP, and API"] --> B["Source-specific adapter"]
    B --> C{"Contract and quality pass?"}
    C -->|"No"| D["Hold for review with a safe reason"]
    C -->|"Deleted"| E["Emit deletion marker"]
    C -->|"Yes"| F["Create versioned parsed document"]
    D --> G["Owner review and replay"]
```

A hold is a governed branch, not a miscellaneous error bucket. Its operational record carries a safe reason and enough lineage to replay from governed storage, but no raw sensitive text. A deletion branch emits a marker instead of parsed content. Only the passing branch creates a versioned parsed document.

### Read the branches, not just the counts

Run the next cell and locate the separate outcomes for stale policy, column layout, duplicate pages, low OCR confidence, deletion, missing territory, and required-field drift. In particular, verify that a missing territory never becomes `worldwide` and that an undocumented field rename never becomes a silent blank.

This is the safety boundary for mapping: uncertainty remains replayable and reviewable, but non-searchable, until its owner resolves it.

In [ ]:
# -- Build DATA-02 mappings and DATA-03 quality evidence ---------------------
def repeated_pdf_pages(record: dict) -> list[int]:
    if record['payload_type'] != 'pdf_pages':
        return []
    seen, duplicates = set(), []
    for page in record['payload']['pages']:
        fingerprint = json.dumps(page.get('text_blocks', []), sort_keys=True)
        if fingerprint in seen:
            duplicates.append(page['page_number'])
        seen.add(fingerprint)
    return duplicates

def local_disposition(record: dict, ocr_threshold: float = 0.80) -> tuple[str, list[str]]:
    reasons, payload = [], record['payload']
    if record['lifecycle']['status'] == 'superseded': reasons.append('not_current')
    if 'deleted' in record['deletion_state']: reasons.append('tombstone_required')
    if record['payload_type'] == 'pdf_columns': reasons.append('layout_parser_required')
    duplicates = repeated_pdf_pages(record)
    if duplicates: reasons.append(f'duplicate_pages:{duplicates}')
    if record['payload_type'] == 'pdf_pages':
        low_ocr = [page['page_number'] for page in payload['pages'] if page.get('ocr_confidence', 1.0) < ocr_threshold]
        if low_ocr: reasons.append(f'low_ocr_pages:{low_ocr}')
    if record['payload_type'] == 'erp_row' and 'territory' in payload and payload['territory'] is None: reasons.append('territory_unknown')
    if record['payload_type'] == 'api_page' and any('status' not in item for item in payload['items']): reasons.append('required_status_schema_drift')
    if any(reason.startswith(('duplicate_pages', 'low_ocr_pages')) for reason in reasons): return 'QUARANTINE_PARSE', reasons
    if 'territory_unknown' in reasons or 'required_status_schema_drift' in reasons: return 'QUARANTINE_CONTRACT', reasons
    if 'tombstone_required' in reasons: return 'TOMBSTONE', reasons
    if 'not_current' in reasons: return 'EXCLUDE_VERSION', reasons
    if 'layout_parser_required' in reasons: return 'CONDITIONAL_PARSE', reasons
    return 'ACCEPT_MAPPING', reasons

mapping_results = {record['record_id']: local_disposition(record) for record in records}
disposition_counts = Counter(result[0] for result in mapping_results.values())
quality_report = {
    'artifact_id': 'DATA-03',
    'claim_class': 'Measured - local fixture',
    'fixture_version': samples['fixture_version'],
    'sample_size': len(records),
    'disposition_counts': dict(sorted(disposition_counts.items())),
    'quarantine_record_ids': sorted(
        record_id for record_id, result in mapping_results.items()
        if result[0] in {'QUARANTINE_PARSE', 'QUARANTINE_CONTRACT'}
    ),
    'limitations': [
        'Synthetic seeded-shape checks are not representative parser benchmarks.',
        'No customer source, Databricks job, or vector index was tested.',
    ],
}
assert mapping_results['REC-RIV-PDF-001'][0] == 'EXCLUDE_VERSION'
assert mapping_results['REC-RIV-PDF-003'][0] == 'CONDITIONAL_PARSE'
assert mapping_results['REC-RIV-PDF-004'][0] == 'QUARANTINE_PARSE'
assert mapping_results['REC-RIV-TEXT-002'][0] == 'TOMBSTONE'
assert mapping_results['REC-RIV-ERP-002'][0] == 'QUARANTINE_CONTRACT'
assert mapping_results['REC-RIV-API-002'][0] == 'QUARANTINE_CONTRACT'
print(f"[Measured - local fixture] DATA-03 sample size: {quality_report['sample_size']}")
print(f"[Measured - local fixture] Dispositions: {quality_report['disposition_counts']}")
print(f"[Measured - local fixture] Quarantine: {quality_report['quarantine_record_ids']}")
print('Prediction resolution: every shortcut violates a different contract boundary.')
print('Limitation: seeded-shape detection is not a parser benchmark.')

### From the Hand Trace to a Repeatable Rule

The first policy record established the order manually. `local_disposition` now applies the same visible sequence across source shapes: inspect lifecycle and deletion state, preserve layout meaning, detect repeated pages, assess readability, and enforce required business fields.

Review five contrasting records against that sequence:

- The stale policy is excluded from the current view without erasing history.
- The repeated-page rights PDF enters parse quarantine; duplication and OCR confidence remain separate reasons.
- The deleted autosave emits a tombstone before its text can be accepted.
- The ERP row with no territory enters contract quarantine rather than receiving a broad rights default.
- The workflow page with the renamed field waits for a versioned contract decision.

A single record may carry several reasons, but its searchable consequence must still be unambiguous. Lifecycle is checked before parse quality because beautifully parsed deleted or superseded text is still not current content.

### Audit the generalization

Trace those five records through `mapping_results` and compare the reasons with the manual sequence above. Then inspect DATA-03: the measured disposition counts describe this frozen sample only, while representative parser accuracy remains explicitly unmeasured.

The function is a transparent teaching model, not a production parser. Production ingestion remains owned by the linked remote pipeline, where these branches need durable review queues, metrics, and replay behavior.

## 3 - Decide What Is Current Before Asking What Looks Similar

The mapping branches answer whether a record can be interpreted. They do not yet answer which version belongs in the current view. Riverside's deleted chapter 38 autosave resembles the current chapter closely enough that content-first deduplication could resurrect the deleted copy or discard the live one.

```mermaid
flowchart LR
    A["Tenant and canonical source location"] --> B["Stable document ID"]
    B --> C["Source version and content fingerprint"]
    C --> D{"Current and active?"}
    D -->|"No"| E["Keep history; exclude or delete downstream copies"]
    D -->|"Yes"| F["Current searchable view"]
```

Identity is therefore scoped before similarity: tenant, canonical source location, stable document ID, source version, and content fingerprint. Immutable versions preserve history; lifecycle state determines the bounded current view. Similarity may suggest a review, but it cannot prove shared owner, access, or lifecycle.

### Try to resurrect the old policy

Set `include_superseded` to `True`. The current-view assertion should fail when the 2024 policy returns. Restore the flag, then confirm that the deleted autosave remains absent while the current chapter survives.

That failure is intentional: overwriting history loses evidence, while merging on similar text loses identity. Versioned history plus a derived current view avoids both errors.

In [ ]:
# -- Measure and check version selection ------------------------------------
def stable_version_key(record: dict) -> tuple[str, str, str, str]:
    return record['tenant_id'], record['document_id'], record['source_version'], record['content_hash']
naive_policy_ids = [record['document_id'] for record in records if record['source_id'] == 'SRC-PDF-POLICY-001']
include_superseded = False  # CHANGE THIS: True should fail the current-view check.
selected_policy_ids = [record['document_id'] for record in records if record['source_id'] == 'SRC-PDF-POLICY-001' and (include_superseded or record['lifecycle']['status'] == 'current') and record['deletion_state'] == 'active']
current_ids = [record['document_id'] for record in records if record['lifecycle']['status'] == 'current' and record['deletion_state'] == 'active']
version_keys = [stable_version_key(record) for record in records]
assert 'DOC-POL-AI-2024' in naive_policy_ids and 'DOC-POL-AI-2024' not in selected_policy_ids
assert 'DOC-POL-AI-2026' in selected_policy_ids
assert 'DOC-MANUSCRIPT-ARIA-038-AUTOSAVE' not in current_ids
assert len(version_keys) == len(set(version_keys))
print(f'[Measured - local fixture] Naive policies: {naive_policy_ids}')
print(f'[Measured - local fixture] Current policies: {selected_policy_ids}')
print('PASS: lifecycle selection excludes stale and deleted records without erasing lineage.')

## 4 - Do Not Advance the Cursor Past an Unknown Contract

The current-view rule is only useful if ingestion sees the complete source. Riverside's first workflow page returns 100 records and a cursor. The second returns 37 more, but renames `status` without a contract version. Stopping after page one silently loses work; permissive parsing silently loses meaning.

```mermaid
flowchart LR
    A["Page 1: 100 records"] --> B{"More pages?"}
    B -->|"Yes"| C["Page 2: 37 records"]
    C --> D{"Required fields match the approved contract?"}
    D -->|"No"| E["Hold page; keep previous sync checkpoint"]
    D -->|"Yes"| F["Replay-safe merge"]
    F --> G["Advance sync checkpoint"]
```

Before running the cell, predict the traversal count: following the cursor should discover 137 records. Then set `accept_undocumented_alias` to `True`. A status value may appear, but the approved contract check must still fail because code convenience is not schema approval.

### The checkpoint rule

Advance sync state only after accepted, replay-safe writes. A timeout is an unknown outcome, not proof of failure; reconcile with a stable business request key before retrying. If page two violates the contract, hold it and keep the previous checkpoint so a corrected contract can replay the complete boundary.

Completeness and exactness travel together: every page must be followed, and every required field must retain approved meaning.

In [ ]:
# -- Measure and check pagination, drift, and idempotency -------------------
workflow_pages = sorted([record for record in records if record['payload_type'] == 'api_page'], key=lambda record: record['payload']['page'])
one_call_count = workflow_pages[0]['payload']['records_returned']
cursor_count = sum(record['payload']['records_returned'] for record in workflow_pages)
required_fields = {'task_id', 'title_id', 'status', 'assigned_user_id', 'updated_at'}
drift = []
for record in workflow_pages:
    for item in record['payload']['items']:
        missing = sorted(required_fields - set(item))
        if missing: drift.append({'record_id': record['record_id'], 'missing': missing, 'unexpected': sorted(set(item) - required_fields)})
accept_undocumented_alias = False  # CHANGE THIS only with versioned approval.
page_two_item = workflow_pages[1]['payload']['items'][0]
status = page_two_item.get('status')
if accept_undocumented_alias: status = status or page_two_item.get('workflow_status')
history = next(record for record in records if record['record_id'] == 'REC-RIV-API-003')
assert workflow_pages[0]['payload']['next_cursor'] == 'cursor-page-2'
assert cursor_count == 137 and len(drift) == 1 and status is None
assert all(attempt['idempotency_key'] is None for attempt in history['payload']['attempts'])
print(f'[Measured - local fixture] One call: {one_call_count}; cursor complete: {cursor_count}; truncation: {(cursor_count-one_call_count)/cursor_count:.1%}')
print(f'[Measured - local fixture] Drift: {drift}')
print('Prediction resolution: traversal finds 137; required-field drift blocks page two.')
print('Limitation: this does not prove API replay, cursor expiry, or Delta merge behavior.')

## 5 - Recheck Authority at the Moment of Retrieval

A well-mapped, current record is still unsafe if the requester no longer has authority. Riverside's disabled contractor remains in an old `ROLE-EDITOR` snapshot, so copied groups alone would grant access after the contract ended.

```mermaid
flowchart LR
    A["Current request context"] --> B{"Identity enabled now?"}
    B -->|"No"| C["Deny and reconcile"]
    B -->|"Yes"| D{"Tenant, region, title, role, and purpose match?"}
    D -->|"No"| C
    D -->|"Yes"| E["Authorized candidate"]
    E --> F["Audit decision without content text"]
```

Current authorization intersects identity state with tenant, region, title, record ACL, role scope, and request purpose. Tenant membership alone does not imply a permitted purpose, and no stale role may override a disabled identity. The audit trail records the decision and reason without copying content text.

### Two predictions, one boundary

First predict the naive result: does the stale role allow the contractor? Then predict the current-context result. Compare both with the cell output, and test the editor against the manuscript, the rights schedule, and an unrelated purpose.

The fixture demonstrates the fail-closed decision order, not live enforcement. Identity-provider freshness and vector-filter enforcement remain external validation requirements.

In [ ]:
# -- Measure and check ACL decisions ----------------------------------------
identity_records = {record['document_id']: record for record in records if record['payload_type'] == 'identity_record'}
manuscript = next(record for record in records if record['document_id'] == 'DOC-MANUSCRIPT-ARIA-037')
rights = next(record for record in records if record['document_id'] == 'DOC-RIGHTS-ARIA-001')
def acl_role_scope(entry: str) -> tuple[str, str | None]:
    role, separator, scope = entry.partition(':')
    return role, scope if separator else None
def authorize(identity_record: dict, record: dict, purpose: str) -> tuple[bool, str]:
    identity = identity_record['payload']
    if not identity.get('enabled', False): return False, 'identity_disabled'
    if record['tenant_id'] not in identity.get('tenant_ids', []): return False, 'tenant_mismatch'
    if record['region'] != identity.get('region_id'): return False, 'region_mismatch'
    title_id = record['payload'].get('title_id')
    if title_id and title_id not in identity.get('title_ids', []): return False, 'title_mismatch'
    if purpose not in {'editorial_retrieval', 'rights_review'}: return False, 'purpose_not_allowed'
    roles = set(identity.get('role_ids', identity.get('direct_role_ids', [])))
    if any(role in roles and (scope is None or scope == title_id) for role, scope in map(acl_role_scope, record['acl'])): return True, 'acl_match'
    return False, 'acl_no_match'
editor, contractor = identity_records['API-USER-EDITOR-017'], identity_records['API-USER-CONTRACTOR-044']
stale_roles = set(contractor['payload']['stale_nested_group_role_ids'])
naive_allowed = any(acl_role_scope(entry)[0] in stale_roles for entry in manuscript['acl'])
decisions = {
    'editor_manuscript': authorize(editor, manuscript, 'editorial_retrieval'),
    'editor_rights': authorize(editor, rights, 'editorial_retrieval'),
    'contractor_manuscript': authorize(contractor, manuscript, 'editorial_retrieval'),
    'wrong_purpose': authorize(editor, manuscript, 'unrelated_analytics'),
}
assert naive_allowed is True
assert decisions['editor_manuscript'] == (True, 'acl_match')
assert decisions['editor_rights'] == (False, 'acl_no_match')
assert decisions['contractor_manuscript'] == (False, 'identity_disabled')
assert decisions['wrong_purpose'] == (False, 'purpose_not_allowed')
print(f'[Measured - local fixture] Naive contractor allowed: {naive_allowed}')
for name, decision in decisions.items(): print(f'[Measured - local fixture] {name}: {decision}')
print('Prediction resolution: stale roles allow; current identity state denies first.')
print('Limitation: this does not prove IdP or vector-filter enforcement.')

## 6 - Prove Absence All the Way Downstream

Current authority can revoke access, but deletion asks for stronger evidence. Removing a source row does not prove that parsed text, chunks, vectors, and index entries disappeared. Riverside needs lineage from the event to every derived copy and proof that the target no longer returns the content.

```mermaid
flowchart LR
    A["Source event or reconciliation"] --> B["Raw version"]
    B --> C["Parsed version"]
    C --> D["Versioned chunks"]
    D --> E["Vector and index records"]
    A --> F{"Delete or revoke?"}
    F -->|"Yes"| G["Deletion marker and derived deletes"]
    G --> H["Negative query or completion receipt"]
    F -->|"No"| I["Advance accepted sync checkpoint"]
```

Overlap catches late arrivals around a checkpoint; reconciliation repairs events that were delayed or missed; replay-safe keys prevent those repairs from multiplying records. A delete or revoke produces derived deletes and completion evidence, while a current sibling version remains intact.

### Break the repair window

Set `overlap_seconds` to zero and observe the failed protection assertion. Restore it, then confirm complete lineage, reconciliation, deletion propagation, and survival of the current chapter. The local dictionary simulation demonstrates state transition only; target deletion still requires completion receipts or a negative query in the governed environment.

Production ownership stays with the Databricks index operations assets. The acceptance condition is downstream absence with evidence, not best-effort file removal.

In [ ]:
# -- Build and check DATA-04 lineage and deletion ---------------------------
required_lineage = {'tenant_id', 'document_id', 'source_uri', 'source_version', 'content_hash', 'acl', 'region', 'classification', 'ingested_at', 'pipeline_version', 'deletion_state'}
lineage_missing = {record['record_id']: sorted(required_lineage - set(record)) for record in records if required_lineage - set(record)}
lineage_coverage = (len(records) - len(lineage_missing)) / len(records)
initial_index = {record['document_id']: record['record_id'] for record in records if record['payload_type'] == 'text'}
index_after_sync = deepcopy(initial_index)
tombstone_ids = [record['document_id'] for record in records if 'deleted' in record['deletion_state']]
for document_id in tombstone_ids: index_after_sync.pop(document_id, None)
overlap_seconds = 300  # CHANGE THIS: 0 removes late-arrival protection.
reconciliation_enabled = True
assert lineage_coverage == 1.0
assert 'DOC-MANUSCRIPT-ARIA-038-AUTOSAVE' in initial_index and 'DOC-MANUSCRIPT-ARIA-038-AUTOSAVE' not in index_after_sync
assert 'DOC-MANUSCRIPT-ARIA-038' in index_after_sync
assert overlap_seconds > 0 and reconciliation_enabled
print(f'[Measured - local fixture] Lineage coverage: {lineage_coverage:.1%}; tombstones: {tombstone_ids}')
print(f'[Measured - local simulation] Before: {sorted(initial_index)}; after: {sorted(index_after_sync)}')
print('External validation required: target deletion, retention, and reconciliation timing.')

## 7 - Assemble Readiness Without Averaging Away Risk

The notebook has now followed records from arrival through ownership, interpretation, current state, access, and deletion. Readiness combines those decisions per source. It does not average them: many clean records cannot cancel one material rights, parser, contract, or access failure.

```mermaid
flowchart LR
    A["Source inventory"] --> B["Mapping and parsing"]
    B --> C["Quality checks"]
    C --> D["Lifecycle, access, and lineage"]
    D --> E{"Does every material gate pass?"}
    E -->|"No"| F["Blocked or excluded"]
    E -->|"Conditions remain"| G["Conditional"]
    E -->|"Yes"| H["Ready for retrieval evaluation"]
```

Each verdict carries blockers, evidence labels, an owner, external checks, and an expiry trigger. `EXCLUDED` keeps prohibited or stale material out of the current view. `BLOCKED` prevents exposure. `CONDITIONAL` permits only the bounded work named in its rationale.

### Read the verdicts source by source

Run DATA-05. Confirm that rights, ERP, workflow, and identity sources are blocked, while policy and manuscript remain conditional. No source becomes production-ready. Where a bounded current view is approved, describe it as `ready for retrieval evaluation`, never `RAG ready`.

That wording protects the next boundary: these checks say whether data may enter retrieval testing, not whether retrieval is relevant, citations are supported, or generated answers are correct. The weakest safety-relevant gate still controls exposure.

In [ ]:
# -- Build and check DATA-05 retrieval readiness ----------------------------
blocking = {'QUARANTINE_PARSE', 'QUARANTINE_CONTRACT'}
conditional = {'CONDITIONAL_PARSE', 'TOMBSTONE', 'EXCLUDE_VERSION'}
source_verdicts = {}
for source_id, source_records in records_by_source.items():
    dispositions = {mapping_results[record['record_id']][0] for record in source_records}
    if source_id == 'SRC-API-IDENTITY-001':
        verdict, rationale = 'BLOCKED', 'Disabled identity and stale nested group require reconciliation.'
    elif dispositions & blocking:
        verdict, rationale = 'BLOCKED', 'At least one sample is in parser or contract quarantine.'
    elif dispositions & conditional:
        verdict, rationale = 'CONDITIONAL', 'Lifecycle, parser, or deletion controls need owner evidence.'
    else:
        verdict, rationale = 'CONDITIONAL', 'Local mapping passes; authority and target enforcement are unvalidated.'
    source_verdicts[source_id] = {'verdict': verdict, 'rationale': rationale, 'external_validation_required': True}
overall_verdict = 'BLOCKED' if any(result['verdict'] == 'BLOCKED' for result in source_verdicts.values()) else 'CONDITIONAL'
assert overall_verdict == 'BLOCKED'
for source_id in ('SRC-PDF-RIGHTS-001', 'SRC-ERP-CATALOG-001', 'SRC-API-WORKFLOW-001', 'SRC-API-IDENTITY-001'): assert source_verdicts[source_id]['verdict'] == 'BLOCKED'
for source_id in ('SRC-PDF-POLICY-001', 'SRC-TEXT-MANUSCRIPT-001'): assert source_verdicts[source_id]['verdict'] == 'CONDITIONAL'
assert all(result['external_validation_required'] for result in source_verdicts.values())
print(f'[Measured - local fixture] Overall DATA-05 verdict: {overall_verdict}')
for source_id, result in source_verdicts.items(): print(f"  {source_id}: {result['verdict']} - {result['rationale']}")
print('Prediction resolution: no production claim exists; every source is conditional or blocked.')
print('Next gate: retrieval and citation evaluation for approved current views only.')

## 8 - Hand Off the Bounded Current View

The local story ends with classification, not deployment. DATA-01 through DATA-05 explain what arrived, why each source is governed or held, and which current records may approach retrieval evaluation. Riverside still needs target-system proof for identity freshness, storage and index controls, deletion, scale, region, and operations.

```mermaid
flowchart LR
    A["Local fixture checks"] --> B["DATA-01 through DATA-05"]
    B --> C["Authorized run record"]
    C --> D["Databricks validation"]
    C --> E["Identity validation"]
    D --> F["Retrieval and citation evaluation"]
    E --> F
    F --> G["Customer readiness decision"]
```

### Package evidence for the next owners

Before handoff, classify every named technique as one of three things: an executable local check, an explained link to its owning implementation, or a named external evidence requirement. Populate the quality report and run record only after an authorized run preserves environment, commit, fixture version, method, observations, and limitations.

Identity freshness, purpose, and title scope move into the identity chapter. Only approved current views move into hybrid-search and RAG evaluation. OCR benchmarks, legal retention, cloud access, networking, scale, cost, region, and target deletion remain with their external owners.

The evidence language must survive that handoff:

- `[Measured - local fixture]` describes deterministic calculations over committed synthetic records.
- `[Modeled]` describes projections from stated assumptions.
- `[Customer-validated]` is reserved for an authorized Riverside decision.
- `[External validation required]` marks behavior the fixture cannot prove.

Local data checks open the retrieval-evaluation gate for a bounded view. They do not close the customer-readiness decision.